In [ ]:
# TV OFF / ACE-Step 1.5 — non-publishing GPU smoke test
# Kaggle's current base image is Python 3.13; upstream ACE-Step currently
# requires Python <3.13. We therefore create an isolated Python 3.12 venv.
import os, sys, json, subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "ACE-Step-1.5"
OUT = WORK / "ace_step_output"
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)

# Confirm that Kaggle has the expected CUDA hardware before installing packages.
import torch
print("Kaggle Python:", sys.version)
print("Kaggle PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing CPU fallback.")
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
for i, name in enumerate(gpu_names):
    print("GPU", i, name, round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2), "GiB")
if not any("T4" in name for name in gpu_names):
    raise RuntimeError(f"Expected Tesla T4; detected {gpu_names}")

# Pin to the actual upstream latest release tag (v0.1.8), not a nonexistent v1.5.2 tag.
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", "v0.1.8",
         "https://github.com/ace-step/ACE-Step-1.5.git", REPO])

# ACE-Step declares Python >=3.11,<3.13. Kaggle's host is Python 3.13,
# so use an isolated 3.12 runtime instead of patching upstream metadata.
uv = Path.home() / ".local" / "bin" / "uv"
if not uv.exists():
    run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"])
if not uv.exists():
    raise RuntimeError("uv executable was not found after installation.")
run([uv, "python", "install", "3.12"])
venv = REPO / ".venv"
run([uv, "venv", "--python", "3.12", str(venv)], cwd=REPO)
python = venv / "bin" / "python"
run([uv, "pip", "install", "--python", str(python), "-e", "."], cwd=REPO)

# Generate using the isolated supported Python runtime.
generation_script = r'''
import json, math, shutil
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from acestep.handler import AceStepHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

OUT = Path("/kaggle/working/ace_step_output")
dit = AceStepHandler()
message, initialized = dit.initialize_service(
    project_root="/kaggle/working/ACE-Step-1.5",
    config_path="acestep-v15-turbo",
    device="cuda",
    offload_to_cpu=True,
    quantization="int8_weight_only",
)
print("ACE-Step initialization:", message, flush=True)
if not initialized:
    raise RuntimeError("ACE-Step DiT initialization failed.")

params = GenerationParams(
    task_type="text2music",
    caption=(
        "Original instrumental downtempo ambient electronic composition, "
        "warm felt piano playing a memorable evolving melody, gentle analog "
        "synth arpeggios, deep rounded bass, layered harmonic progression, "
        "subtle rhythmic pulse, changing musical sections, expressive dynamics, "
        "professional studio production, no vocals, no speech, no sound effects"
    ),
    lyrics="[Instrumental]",
    instrumental=True,
    vocal_language="unknown",
    bpm=82,
    duration=30,
    inference_steps=8,
    shift=3.0,
    seed=20261120,
    thinking=False,
    use_cot_metas=False,
    use_cot_caption=False,
    use_cot_language=False,
)
config = GenerationConfig(
    batch_size=1,
    allow_lm_batch=False,
    use_random_seed=False,
    seeds=[20261120],
    audio_format="wav",
)
result = generate_music(dit, None, params, config, save_dir=str(OUT))
if not result.success:
    raise RuntimeError(f"ACE-Step generation failed: {result.error}")
if not result.audios:
    raise RuntimeError("ACE-Step returned success but no audio output.")

source = Path(result.audios[0]["path"])
target = OUT / "tv_off_ace_step_t4_test.wav"
if not source.is_file():
    raise RuntimeError(f"Audio file not found: {source}")
if source.resolve() != target.resolve():
    shutil.copy2(source, target)

info = sf.info(str(target))
audio, sr = sf.read(str(target), always_2d=True, dtype="float32")
peak = float(np.max(np.abs(audio)))
rms = float(np.sqrt(np.mean(np.square(audio))))
duration = float(info.duration)
if not 25 <= duration <= 35:
    raise RuntimeError(f"Unexpected audio duration: {duration:.2f}s")
if not np.isfinite(audio).all():
    raise RuntimeError("Audio contains NaN or infinite samples.")
if peak < 0.01 or peak > 1.001:
    raise RuntimeError(f"Invalid peak level: {peak:.5f}")
if rms < 0.001:
    raise RuntimeError(f"Audio RMS too low: {rms:.6f}")

manifest = {
    "status": "ok",
    "provider": "ACE-Step-1.5",
    "release": "v0.1.8",
    "model": "acestep-v15-turbo",
    "python": __import__("sys").version,
    "gpu": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "cuda": True,
    "duration_seconds": round(duration, 3),
    "sample_rate": int(sr),
    "channels": int(info.channels),
    "subtype": info.subtype,
    "peak": round(peak, 6),
    "rms": round(rms, 6),
    "seed": 20261120,
    "instrumental": True,
    "publishing_enabled": False,
    "audio_file": target.name,
}
(OUT / "ace_step_smoke_result.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("TEST PASSED — audio generated and validated; nothing was published.")
'''
script_path = WORK / "run_ace_step_smoke.py"
script_path.write_text(generation_script, encoding="utf-8")
run([python, str(script_path)])
print("ACE-Step test process finished.")
